# Кандидатогенерация для категории «Услуги» (Avito, тестовое задание)

**Задача:** по поисковому запросу вернуть до 50 `item_id` объявлений-кандидатов
из корпуса `benchmark_items.parquet` (189 212 объявлений), которые дальше пойдут
на этап ранжирования. Метрика — **Recall@50** (важна полнота, а не порядок).

**Общий подход:** классический sparse-retrieval (TF-IDF + косинусное сходство).
Обоснование выбора — ниже, по ходу решения, с явной локальной валидацией
качества до того, как тратить попытки сдачи на платформе.

**Открытые библиотеки:** `pandas`, `pyarrow`, `numpy`, `scipy`, `scikit-learn`
(всё стандартное, CPU-only, без обращений к внешним API — решение
воспроизводится полностью локально).

**Структура ноутбука:**
1. Данные и EDA (ключевые находки)
2. Локальная валидация (честная оценка Recall@50 на реальном корпусе)
3. Подбор признаков (ablation) и обоснование финальной конфигурации
4. Финальная модель и генерация `answer.csv`
5. Проверка формата перед сдачей
6. Анализ ошибок


## 1. Данные и EDA

In [1]:
import gc
import re
import time
import pickle

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer

DATA_DIR = "data"
OUT_DIR = "outputs"

# train.parquet тяжёлый (полные тексты объявлений на каждую строку) -- нигде
# дальше не читаем его целиком, только нужные колонки под конкретную задачу,
# чтобы не раздувать память на однопроцессорной машине с ограниченным RAM.
train_meta = pq.ParquetFile(f"{DATA_DIR}/train.parquet").metadata
bq = pd.read_parquet(f"{DATA_DIR}/benchmark_queries.parquet")
bi = pd.read_parquet(f"{DATA_DIR}/benchmark_items.parquet")

print("train: строк =", train_meta.num_rows, ", колонок =", train_meta.num_columns)
print("benchmark_queries:", bq.shape)
print("benchmark_items:", bi.shape)


ModuleNotFoundError: No module named 'numpy'

**Ключевые находки EDA** (полный разбор был на этапе исследования, здесь —
проверка фактов, которые определили дизайн решения):

- `search_category` практически константа: 99.99% строк в train и в
  benchmark_queries имеют одно и то же значение (макрокатегория «Услуги»).
  Использовать её как фильтр бессмысленно — почти нечего фильтровать.
- Реальная суб-категоризация («Красота, здоровье», «Грузоперевозки» и т.п.)
  спрятана в неструктурированном тексте полей `*_infm_params_text` — это
  конкатенация меток и значений без чётких разделителей (не JSON, без
  двоеточий). Схема меток нигде не документирована.
- `item_id` — 16-символьная hex-строка, полностью валидна во всех файлах.


In [ ]:
hex16 = re.compile(r'^[0-9a-f]{16}$')
assert bi["item_id"].astype(str).apply(lambda x: bool(hex16.match(x))).all()

# читаем только нужные колонки train, не всю таблицу целиком (экономия памяти)
train_small = pq.read_table(f"{DATA_DIR}/train.parquet", columns=["item_id", "search_category"]).to_pandas()
assert train_small["item_id"].astype(str).apply(lambda x: bool(hex16.match(x))).all()

print("search_category в train (доля):")
print((train_small["search_category"].value_counts(normalize=True) * 100).round(3).head())
print()
print("уникальных item_category_id в корпусе объявлений:", bi["item_category_id"].nunique())
print("уникальных item_microcat_id в корпусе объявлений:", bi["item_microcat_id"].nunique())
print("(microcat — потенциально сильный сигнал, но у запроса нет прямого аналога, "
      "только текст фильтров — оставляем как идею для будущего улучшения)")

del train_small
gc.collect()


search_category в train (доля):
search_category
114    99.992
0       0.007
81      0.000
10      0.000
Name: proportion, dtype: float64

уникальных item_category_id в корпусе объявлений: 47
уникальных item_microcat_id в корпусе объявлений: 752
(microcat — потенциально сильный сигнал, но у запроса нет прямого аналога, только текст фильтров — оставляем как идею для будущего улучшения)


0

**Решение по `*_infm_params_text`:** не парсим регулярными выражениями на
структурированные пары «метка-значение» — без документированной схемы это
хрупко и может незаметно всё сломать. Вместо этого просто добавляем текст
как есть в общий текстовый блок для TF-IDF: совпадающие слова
(«Красота», «здоровье», «Грузоперевозки») и так дают нужный сигнал через
обычное текстовое сходство, без риска криво распарсить.


## 2. Локальная валидация

`train.parquet` — это пары «запрос → выбранное объявление», а не запрос с
явным `query_id`. Один и тот же текст запроса с разными фильтрами
(локация, доставка, категория) — это **разные** запросы. Поэтому группируем
не по тексту, а по полной комбинации всех `search_*` полей.

Чтобы оценка Recall@50 была честной и соответствовала реальным условиям
задачи, берём в валидацию только те query-группы, у которых **все**
релевантные объявления присутствуют в реальном корпусе `benchmark_items`
(иначе часть релевантных объявлений в принципе недостижима, и recall будет
занижен не из-за качества модели, а из-за устройства выборки).


In [ ]:
search_cols = ["search_query", "search_location_id", "search_is_delivery_search",
               "search_infm_params_text", "search_category"]

bi_ids = set(bi["item_id"])

# читаем только search_* + item_id -- полный train с текстами объявлений
# в память не грузим вообще, он тут не нужен
train_search = pq.read_table(f"{DATA_DIR}/train.parquet", columns=search_cols + ["item_id"]).to_pandas()

grouped = train_search.groupby(search_cols, dropna=False)["item_id"].apply(set).reset_index()
grouped.rename(columns={"item_id": "relevant_items"}, inplace=True)
grouped["n_relevant"] = grouped["relevant_items"].apply(len)
grouped["n_in_corpus"] = grouped["relevant_items"].apply(lambda s: len(s & bi_ids))
grouped["all_in_corpus"] = grouped["n_relevant"] == grouped["n_in_corpus"]

local_val = grouped[grouped["all_in_corpus"]].drop(columns=["n_in_corpus", "all_in_corpus"]).reset_index(drop=True)
print("query-групп всего:", len(grouped))
print("групп с полным покрытием корпусом (используем для валидации):", len(local_val))

del train_search, grouped
gc.collect()


query-групп всего: 354463
групп с полным покрытием корпусом (используем для валидации): 18436


0

## 3. Подбор признаков (ablation)

Метрика — Recall@50, поэтому задача читается как классический
информационный поиск: нужно не идеально ранжировать, а не терять нужное
объявление среди топ-50. Базовый, проверенный и разрешённый условиями метод
для этого — **TF-IDF + косинусное сходство**.

Стоп-слова — захардкоженный список (без обращения к `nltk_data` по сети в
рантайме, чтобы решение гарантированно воспроизводилось офлайн).

В ходе разработки был прогнан более широкий перебор комбинаций полей
(результаты приведены в таблице ниже для прозрачности процесса). Здесь для
экономии времени выполнения ноутбука воспроизводим только две ключевые
точки — минимальный бейзлайн и финальную конфигурацию — этого достаточно,
чтобы показать эффект и не гонять весь перебор заново.

| Конфигурация                              | Recall@50 (локально) |
|--------------------------------------------|----------------------|
| только title                               | 0.219 |
| title + params                             | 0.218 |
| title×2 + params + полное описание         | 0.248 |
| title×2 + params + описание(300 симв.)     | 0.256 |
| **title×3 + params + описание(300 симв.)** | **0.258** |
| title×4 + params + описание(300 симв.)     | 0.258 (плато) |
| title×3 + params + descr(300), min_df=1    | 0.257 (хуже, больше словарь) |
| то же + биграммы (1,2)                     | 0.255 (хуже, в 2 раза медленнее) |

Обрезка описания до 300 символов не просто ускоряет счёт, а **улучшает**
recall — длинный хвост описания (график работы, условия оплаты) в основном
шум, который размывает сигнал. Вес title дальше x3 уже не даёт прироста —
остановились на нём, не усложняя дальше.


In [ ]:
STOPWORDS = [
    "и","в","во","не","что","он","на","я","с","со","как","а","то","все","она",
    "так","его","но","да","ты","к","у","же","вы","за","бы","по","только","ее",
    "мне","было","вот","от","меня","еще","нет","о","из","ему","теперь","когда",
    "даже","ну","вдруг","ли","если","уже","или","ни","быть","был","него","до",
    "вас","нибудь","опять","уж","вам","сказал","ведь","там","потом","себя",
    "ничего","ей","может","они","тут","где","есть","надо","ней","для","мы",
    "тебя","их","чем","была","сам","чтоб","без","будто","человек","чего",
    "раз","тоже","себе","под","будет","ж","тогда","кто","этот","того","потому",
    "этого","какой","совсем","ним","здесь","этом","один","почти","мой","тем",
    "чтобы","нее","были","куда","зачем","всех","никогда","можно","при","наконец",
    "два","об","другой","хоть","после","над","больше","тот","через","эти","нас",
    "про","всего","них","какая","много","разве","три","эту","моя","впрочем",
    "хорошо","свою","этой","перед","иногда","лучше","чуть","том","нельзя",
    "такой","им","более","всегда","конечно","всю","между",
]

TOP_K = 50
BATCH = 200

def build_item_text(bi_df, title_repeat, descr_maxlen):
    title = bi_df["item_title_raw"].fillna("")
    params = bi_df["item_infm_params_text"].fillna("")
    text = (title + " ") * title_repeat + params
    if descr_maxlen > 0:
        descr = bi_df["item_description_raw"].fillna("").str.slice(0, descr_maxlen)
        text = text + " " + descr
    return text

def top50_per_query(item_matrix_T, query_matrix, fallback_idx=None):
    """Возвращает список массивов индексов объявлений (топ-50) для каждой строки query_matrix."""
    n = query_matrix.shape[0]
    out = []
    for start in range(0, n, BATCH):
        end = min(start + BATCH, n)
        sim = item_matrix_T.T.dot(query_matrix[start:end].T).tocsc()
        for k in range(sim.shape[1]):
            col = sim.getcol(k)
            if col.nnz == 0:
                idx = fallback_idx if fallback_idx is not None else np.array([], dtype=int)
            else:
                data, rows = col.data, col.indices
                top_local = (np.argpartition(-data, TOP_K - 1)[:TOP_K]
                            if len(data) > TOP_K else np.arange(len(data)))
                top_local = top_local[np.argsort(-data[top_local])]
                idx = rows[top_local]
            out.append(idx)
    return out

def evaluate_recall(item_ids, pred_idx_list, relevant_items_list):
    recalls = []
    for idx, relevant in zip(pred_idx_list, relevant_items_list):
        hit = len(set(item_ids[idx]) & set(relevant))
        recalls.append(hit / len(relevant))
    return float(np.mean(recalls))


In [ ]:
# --- Бейзлайн для сравнения: только title ---
t0 = time.time()
item_text_baseline = build_item_text(bi, title_repeat=1, descr_maxlen=0)
vec_baseline = TfidfVectorizer(lowercase=True, stop_words=STOPWORDS,
                                sublinear_tf=True, min_df=2, max_df=0.5, dtype=np.float32)
item_matrix_baseline = vec_baseline.fit_transform(item_text_baseline)
item_matrix_baseline_T = item_matrix_baseline.T.tocsr()

query_text_val = local_val["search_query"].fillna("") + " " + local_val["search_infm_params_text"].fillna("")
query_matrix_baseline = vec_baseline.transform(query_text_val)

item_ids = bi["item_id"].to_numpy()
pred_baseline = top50_per_query(item_matrix_baseline_T, query_matrix_baseline)
recall_baseline = evaluate_recall(item_ids, pred_baseline, local_val["relevant_items"])
print(f"Бейзлайн (только title): recall@50 = {recall_baseline:.4f}  ({time.time()-t0:.1f}с)")

del item_text_baseline, vec_baseline, item_matrix_baseline, item_matrix_baseline_T
del query_matrix_baseline, pred_baseline
gc.collect()


Бейзлайн (только title): recall@50 = 0.2175  (27.4с)


0

In [ ]:
# --- Финальная конфигурация: title x3 + params + описание(300 симв.) ---
t0 = time.time()
item_text_final = build_item_text(bi, title_repeat=3, descr_maxlen=300)
vectorizer = TfidfVectorizer(lowercase=True, stop_words=STOPWORDS,
                              sublinear_tf=True, min_df=2, max_df=0.5, dtype=np.float32)
item_matrix = vectorizer.fit_transform(item_text_final)
item_matrix_T = item_matrix.T.tocsr()

query_matrix_val = vectorizer.transform(query_text_val)
pred_final = top50_per_query(item_matrix_T, query_matrix_val)
recall_final = evaluate_recall(item_ids, pred_final, local_val["relevant_items"])
print(f"Финальная конфигурация: recall@50 = {recall_final:.4f}  ({time.time()-t0:.1f}с)")
print(f"Размер словаря: {len(vectorizer.vocabulary_)}")

assert recall_final > recall_baseline, "финальная конфигурация должна быть лучше бейзлайна"


Финальная конфигурация: recall@50 = 0.2582  (36.5с)
Размер словаря: 159309


## 4. Финальная модель и генерация `answer.csv`

Векторайзер и матрица объявлений (`vectorizer`, `item_matrix_T`) уже
обучены на полном корпусе `benchmark_items` в предыдущем шаге — переиспользуем
их напрямую, без повторного обучения.

**Фолбэк на случай нулевого TF-IDF-вектора запроса.** При генерации ответа на
реальный `benchmark_queries` обнаружились запросы с редкими/опечатанными
словами («сваршики» вместо «сварщики», «кожевник», «сабутыльник»,
«подручный»), которых нет в словаре объявлений вообще — TF-IDF возвращает
нулевой вектор, кандидатов нет. Вместо пустого ответа отдаём топ-50 самых
популярных объявлений (по `рейтинг × log(число отзывов)`) — стандартный
приём для cold-start/no-match случаев в поисковых системах.


In [ ]:
popularity_score = bi["item_rating"].fillna(0) * np.log1p(bi["item_rating_reviews_count"].fillna(0))
fallback_idx = popularity_score.to_numpy().argsort()[::-1][:TOP_K]

query_text_bench = bq["search_query"].fillna("") + " " + bq["search_infm_params_text"].fillna("")
query_matrix_bench = vectorizer.transform(query_text_bench)

pred_bench = top50_per_query(item_matrix_T, query_matrix_bench, fallback_idx=fallback_idx)
answers = [" ".join(item_ids[idx]) for idx in pred_bench]

result = pd.DataFrame({"query_id": bq["query_id"].astype(str), "answer": answers})
import os
os.makedirs(OUT_DIR, exist_ok=True)
result.to_csv(f"{OUT_DIR}/answer.csv", index=False)
print("answer.csv сохранён, строк:", len(result))
result.head(3)


answer.csv сохранён, строк: 2452


,query_id,answer
0,70DfDUpwjxB4lzFd,d722bcda1a555091 355392014208b7bf 603623b4bd9e...
1,JTrdTaZJvSiLPkXj,413f6384e3c07b64 4a435cc0a254c1fd 8703b2a0442c...
2,LZCZNoVG4AFUkVRJ,3f89b8062dc85f1c d8fce513e4f000a7 168a9207e80b...


## 5. Проверка формата перед сдачей

In [ ]:
ans = pd.read_csv(f"{OUT_DIR}/answer.csv", dtype=str, keep_default_na=False, na_filter=False)

assert list(ans.columns) == ["query_id", "answer"], "должно быть ровно 2 колонки"
assert len(ans) == len(bq), "строка на каждый query_id из benchmark_queries"
assert ans["query_id"].duplicated().sum() == 0, "не должно быть дублей query_id"
assert set(ans["query_id"]) == set(bq["query_id"].astype(str)), "query_id должны совпадать 1-в-1 с benchmark_queries"
assert (ans["answer"] == "").sum() == 0, "не должно быть пустых ответов"

for s in ans["answer"]:
    ids = s.split(" ")
    assert len(ids) <= 50, "не больше 50 item_id в строке"
    assert len(set(ids)) == len(ids), "без повторов внутри строки"
    assert all(hex16.match(i) for i in ids), "item_id должны быть в формате 0-9a-f, 16 символов"
    assert all(i in bi_ids for i in ids), "все item_id должны существовать в benchmark_items"

print("Все проверки формата пройдены.")


Все проверки формата пройдены.
